In [ ]:
import sys
sys.path.append('../')

# import os
# os.chdir("../")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.datagen import load_s3_data_as_df, load_local_data_as_df
from utils.utils import json_numpy_serialzer
from utils.logging import LOGGER

from sanitisation_techniques.sanitiser import SanitiserNHS
from generative_models.data_synthesiser import (IndependentHistogram,
                                                BayesianNet,
                                                PrivBayes,
                                                Cvine,
                                                CvineSensitive,
                                                IMRV)
from generative_models.CTGAN import CTGAN
from generative_models.TVAE import TVAE
from predictive_models.predictive_model import RandForestClassTask, LogRegClassTask, LinRegTask

SEED = 42

In [ ]:
# Read real data (look at a more realistic example!!)
s3name = "../data/simulated_data/real_data"
data, metadata = load_local_data_as_df(s3name)
sensitive = ["x1", "x6", "x12"]

In [ ]:
# Train-test-split
train_ind = np.random.choice([True, False], data.shape[0], replace=True, p=[0.7,0.3])
data_train = data.iloc[train_ind]
data_test = data.iloc[~train_ind]

In [ ]:
# Initialize random forest classifier           
rf = RandForestClassTask(metadata=metadata, labelCol="y")

In [ ]:
# Train on real test on real performance
rf.train(data_train)
rf.get_metrics(data_test)

In [ ]:
# Train on synthetic test on synthetic performance
gmList = [PrivBayes(metadata, 25, 1, 0.1),
          Cvine(metadata, "all"),
        #   CvineSensitive(metadata, "all", sensitive, 5),
          IMRV(metadata, "all", sensitive, 0.5),
          CTGAN(metadata, 400,100),
          TVAE(metadata, 800, 100)];

In [ ]:
acc_results = {}
auc_results = {}
for gm in gmList:
    print(f"Running utility evaluation for {gm.__name__}")
    gm.fit(data_train)
    synData = [gm.generate_samples(500) for _ in range(50)]
    acc = auc = np.zeros(len(synData))
    for i, synDat in enumerate(synData):
        rf.train(synDat)
        metrics = rf.get_metrics(data_test)
        acc[i] = metrics['Accuracy'] 
        auc[i] = metrics["AUC-ROC"]
    acc_results[gm.__name__] = acc
    auc_results[gm.__name__] = auc

In [ ]:
auc_results

In [ ]:
pd.DataFrame(auc_results).boxplot(rot=90, grid=False)

In [ ]:
metrics